### Full batch

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime

In [0]:
full_table = ['customers','products','categories']

for table in full_table:
    path = f'/Volumes/main/volume/task/project/{table}/'
    file = dbutils.fs.ls(path)
    df = spark.read.option('header',True).csv(file[-1].path)
    df.write.mode('overwrite').saveAsTable(f'main.raw_task.{table}')

### incremental load

In [0]:
incremental_table = ['orders','payments']

for table in incremental_table:
    path = f'/Volumes/main/volume/task/project/{table}/'
    files = dbutils.fs.ls(path)

    if not files: continue

    watermark_table = f'main.watermark.watermark_{table}'
    target_table = f'main.raw_task.{table}'
    watermark_timestamp = spark.sql(f'select watermark from {watermark_table}').collect()[0][0]
    latest_timestamp = watermark_timestamp
    new_files=[]

    for f in files:
        last_timestamp = datetime.fromtimestamp(f.modificationTime/1000)
        if last_timestamp > watermark_timestamp:
            new_files.append(f.path)
            if last_timestamp > latest_timestamp:
                latest_timestamp = last_timestamp
    if not new_files: continue

    df = spark.read.option('header',True).csv(new_files)
    pk = table[:-1] + '_id' 
    df = df.dropDuplicates([pk])
    if not spark.catalog.tableExists(f'main.raw_task.{table}'):
        df.write.mode('overwrite').saveAsTable(target_table)
    else:
        df.createOrReplaceTempView(f'{table}_update')

        spark.sql(f"""
              merge into {target_table} as target
              using {table}_update as source
              on target.{pk} = source.{pk}
              when matched then update set *
              when not matched then insert *
              """)
    
  
        spark.sql(f'update {watermark_table} set watermark = timestamp "{latest_timestamp}"')

